# 5e — S2D Physical-Consistency Diagnostics
## JRA55\_FOSIRL vs Reanalysis: Process Relationships and Energy Closure

This notebook evaluates whether the **JRA55_FOSIRL** and **Reanalysis**
hindcasts preserve physically consistent land-atmosphere and ocean-atmosphere
coupling across lead time.

### Key Process Diagnostics

1. **Flux Partitioning**: Evaporative Fraction ($EF$) & Bowen Ratio ($BR$).
2. **Land-Atmosphere Coupling**: Soil moisture vs LHFLX ($\beta_{SM \to LH}$) and TREFHT ($\beta_{SM \to T}$).
3. **Precipitation-Soil Moisture Response**: Daily lagged response slope $\beta_{P \to \Delta SM}(\ell)$ (lags 0–7 days).
4. **Ocean-Atmosphere Contrast**: Air-sea temperature difference $\Delta T_{AO} = TREFHT - SST$.
5. **Surface Energy Budget**: Apparent energy residual $R_{\text{apparent}} = FSNS - FLNS - LHFLX - SHFLX$.

> ⚠️ **S2D Experiment Labels**: All figures and tables use neutral labels:
> `JRA55_FOSIRL`, `Reanalysis`, and `JRA55_FOSIRL − Reanalysis`.


In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import os
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

import sys
import esp_lab

# Development-checkout fallback: a clean install exposes ``workflows`` directly.
REPO_ROOT = Path(esp_lab.__file__).resolve().parent.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from workflows.diagnostics.physical_consistency import config as configuration
from workflows.diagnostics.physical_consistency import energy_budget
from workflows.diagnostics.physical_consistency import flux_partitioning
from workflows.diagnostics.physical_consistency import land_coupling
from workflows.diagnostics.physical_consistency import ocean_coupling
from workflows.diagnostics.physical_consistency import precip_sm_response as precipitation_soil_moisture
from workflows.diagnostics.physical_consistency import run_physical as physical_workflow

from esp_lab.diagnostics.physical_core import (
    compute_evaporative_fraction, compute_bowen_ratio,
    compute_sst_trefht_contrast, integrate_soil_moisture,
    compute_coupling_slope, compute_precip_sm_lag_response,
    compute_apparent_energy_residual, bootstrap_physical_metric_ci,
)
from esp_lab.utils.dask_utils import DaskConfig, get_cluster_client, close_cluster

PHYSICAL_DIR = Path(configuration.__file__).resolve().parent
print('Physical Consistency imports OK')


## Dask Setup

In [ ]:
machine_env = os.environ.get('CLUSTER_TYPE', 'local')
dask_cfg = DaskConfig(cluster_type=machine_env, workers=8, cores=4, memory='16GB')
cluster, client = get_cluster_client(dask_cfg)
print(client)


## Configuration

In [ ]:
PILOT_ONLY  = True
SEASON      = 'may'      # 'may' or 'nov'
FREQUENCY   = 'daily'    # 'daily' or 'monthly'
S2D_DIAG_ROOT = Path(os.environ.get('ESP_LAB_S2D_DIAG_ROOT', '/global/cfs/cdirs/e3sm/S2S2D/s2d_diag'))
FIGURE_ROOT = Path(os.environ.get('ESP_LAB_FIGURE_OUTDIR', '/global/cfs/cdirs/e3sm/www/zhan391/esp-lab_diag'))
OUTPUT_ROOT = S2D_DIAG_ROOT / 'multimodel' / 'physical_consistency'
FIGURE_OUTDIR = FIGURE_ROOT / 'physical_consistency'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
FIGURE_OUTDIR.mkdir(parents=True, exist_ok=True)

print('Frequency :', FREQUENCY)
print('Season    :', SEASON)


## 1. Flux Partitioning: Evaporative Fraction ($EF$) & Bowen Ratio ($BR$)

Calculates:
$$EF(W) = \frac{\sum_{d \in W} LHFLX}{\sum_{d \in W} (LHFLX + SHFLX)}$$
$$BR(W) = \frac{\sum_{d \in W} SHFLX}{\sum_{d \in W} LHFLX}$$

*Ratios are evaluated on window-summed fluxes first to avoid unstable daily ratios.*


In [ ]:
%%time
# Demo with synthetic arrays for cell execution check
lat = np.linspace(-90, 90, 5)
lon = np.linspace(-180, 180, 5)
days = list(range(1, 85))
rng = np.random.default_rng(42)

lh_ref  = xr.DataArray(rng.random((2, 3, 84, 5, 5)) * 50 + 20, dims=['Y','M','d','lat','lon'], coords={'d': days, 'lat': lat, 'lon': lon})
sh_ref  = xr.DataArray(rng.random((2, 3, 84, 5, 5)) * 30 + 10, dims=['Y','M','d','lat','lon'], coords={'d': days, 'lat': lat, 'lon': lon})
lh_test = lh_ref + 5.0
sh_test = sh_ref - 2.0

fp_res = flux_partitioning.run_flux_partitioning(lh_ref, sh_ref, lh_test, sh_test, configuration.DAILY_WINDOWS)
for win, res in fp_res.items():
    print(f'  {win:<15s} mean EF(JRA55_FOSIRL)={float(res["ef_test"].mean()):.3f}  mean EF(Reanalysis)={float(res["ef_ref"].mean()):.3f}  QC_masked={res["qc_masked_frac"]:.1%}')


## 2. Soil Moisture–Atmosphere Coupling ($\beta_{SM \to LH}$ & $\beta_{SM \to T}$)

Linear regression slopes:
$$\beta_{SM \to LH} = \frac{\text{Cov}(SM', LHFLX')}{\text{Var}(SM')}$$
$$\beta_{SM \to T} = \frac{\text{Cov}(SM', TREFHT')}{\text{Var}(SM')}$$


In [ ]:
%%time
sm_ref  = xr.DataArray(rng.random((2, 3, 84, 5, 5)) * 0.1 + 0.2, dims=['Y','M','d','lat','lon'], coords={'d': days, 'lat': lat, 'lon': lon})
t_ref   = xr.DataArray(rng.random((2, 3, 84, 5, 5)) * 5 + 15, dims=['Y','M','d','lat','lon'], coords={'d': days, 'lat': lat, 'lon': lon})
sm_test = sm_ref + 0.02
t_test  = t_ref - 0.5

lc_res = land_coupling.run_land_coupling(sm_ref, lh_ref, t_ref, sm_test, lh_test, t_test, configuration.DAILY_WINDOWS)
for win, res in lc_res.items():
    print(f'  {win:<15s} mean β(SM->LH) JRA55_FOSIRL={float(res["slope_lh_test"].mean()):.3f}')


## 3. Daily Precipitation–Soil Moisture Lag Response $\beta_{P \to \Delta SM}(\ell)$

Computes soil moisture tendency $\Delta SM_d = SM_{d+1} - SM_d$ and lag response
slope $\beta(\ell)$ for lags $\ell = 0 \dots 7$ days.


In [ ]:
%%time
pr_ref  = xr.DataArray(rng.random((2, 3, 84, 5, 5)) * 5, dims=['Y','M','d','lat','lon'], coords={'d': days, 'lat': lat, 'lon': lon})
pr_test = pr_ref + 0.5

psr_res = precipitation_soil_moisture.run_precip_sm_response(pr_ref, sm_ref, pr_test, sm_test, max_lag=7)
beta_diff = psr_res['paired_diff_lag']
fig, ax = plt.subplots(figsize=(6, 3.5), dpi=120)
ax.plot(beta_diff.lag, beta_diff.mean(['lat','lon']).values, marker='o', color='purple', lw=2)
ax.axhline(0, color='k', ls='--', lw=0.8)
ax.set_xlabel('Lag ℓ (days)', fontsize=11)
ax.set_ylabel('Δβ(P → ΔSM)', fontsize=11)
ax.set_title('Precipitation to soil-moisture lag response difference', fontsize=12)
plt.tight_layout()
plt.show()


## 4. Ocean–Atmosphere Coupling: Air-Sea Temperature Contrast ($\Delta T_{AO}$)

$$\Delta T_{AO} = TREFHT - SST$$


In [ ]:
%%time
sst_ref  = t_ref - 2.0
sst_test = t_test - 1.8

oc_res = ocean_coupling.run_ocean_coupling(t_ref, sst_ref, t_test, sst_test, configuration.DAILY_WINDOWS)
for win, res in oc_res.items():
    print(f'  {win:<15s} mean ΔT_AO(JRA55_FOSIRL)={float(res["dt_ao_test"].mean()):.3f} °C')


## 5. Surface Energy Budget: Apparent Residual ($R_{\text{apparent}}$)

$$R_{\text{apparent}} = FSNS - FLNS - LHFLX - SHFLX$$


In [ ]:
%%time
fsns_ref = xr.DataArray(rng.random((2, 3, 84, 5, 5)) * 100 + 150, dims=['Y','M','d','lat','lon'], coords={'d': days, 'lat': lat, 'lon': lon})
flns_ref = xr.DataArray(rng.random((2, 3, 84, 5, 5)) * 30 + 40, dims=['Y','M','d','lat','lon'], coords={'d': days, 'lat': lat, 'lon': lon})
fsns_test = fsns_ref + 2.0
flns_test = flns_ref - 1.0

eb_res = energy_budget.run_energy_budget(fsns_ref, flns_ref, lh_ref, sh_ref, fsns_test, flns_test, lh_test, sh_test, configuration.DAILY_WINDOWS)
for win, res in eb_res.items():
    print(f'  {win:<15s} mean R_apparent(JRA55_FOSIRL)={float(res["res_test"].mean()):.3f} W/m²')


## Shutdown

In [ ]:
close_cluster(cluster, client)
